# 하루치 전체 태그 추출 (CSV)

`Whole_Data_Extractor.py` 의 `LogExtractor` 로, 특정 태그 목록이 아니라 **그 날 실제로 값이 있던 태그 전체**를 CSV 로 받는다.

동작 방식:
1. `discover_tags()` — measurement `plc_line2` 안에서 그 날 값이 있었던 `tag_name` 을 가볍게(피벗 없이) 먼저 조회한다. 사람이름 태그와 IL주소(T1254 등) 중복 시리즈를 구분하지 않고 **있는 그대로 전부** 가져온다.
2. `extract_day_to_csv()` — 그 목록을 **1시간 chunk**(`CHUNK_HOURS = 1`)로 나눠 받으면서, **청크 하나를 처리할 때마다 바로 CSV 에 이어 쓰고 메모리에서 비운다.**

**왜 한 번에 concat 해서 저장하지 않았나.** 태그가 1만개 규모라 1시간 청크 하나가 실측 약 480MB — 24개를 다 메모리에 들고 있다가 합치면(예전 `get_data` 방식) 이 PC 가용 메모리(테스트 시점 약 12GB)를 넘길 수 있다. 그래서 청크마다 바로 디스크에 쓴다. 대신 ffill 이 청크 경계를 넘어가도록, 직전 청크의 마지막 값을 다음 청크에 이어붙인 뒤 채운다 (`extract_day_to_csv` 안의 `last_state`).

**1번 셀 결과(태그 개수)를 먼저 확인한 뒤에 2번 셀(실제 추출)을 실행하는 걸 권장한다.** 2026-09-20 하루 기준 실측: 태그 10,517개, 1시간당 약 260MB CSV — 하루 전체면 대략 6GB 안팎이 될 수 있다.

In [ ]:
import os
import sys

ROOT_DIR = os.path.dirname(os.getcwd())  # parkkt/ 의 부모 = 저장소 루트
sys.path.insert(0, os.getcwd())

from Whole_Data_Extractor import LogExtractor

ENV_PATH = os.path.join(ROOT_DIR, ".env")  # .env 는 저장소 루트에 있음
extractor = LogExtractor(env_path=ENV_PATH)

## 1. 태그 개수 먼저 확인 (가벼운 쿼리)

피벗 없이 `tag_name` 만 distinct 로 조회한다. 여기서 나온 개수를 보고 다음 셀을 실행할지 판단한다.

In [ ]:
TARGET_DATE = "2026-09-20"  # KST 기준 하루 (00:00 ~ 다음날 00:00)

start_time = f"{TARGET_DATE} 00:00:00"
import pandas as pd
from datetime import timedelta
end_time = (pd.to_datetime(TARGET_DATE) + timedelta(days=1)).strftime("%Y-%m-%d 00:00:00")

discovered_tags = extractor.discover_tags(start_time, end_time)
print(f"\n총 {len(discovered_tags)}개 태그, 1시간 chunk로 24번 나눠서 CSV 에 이어 쓸 예정")

## 2. 실제 추출 (1시간 chunk, 청크마다 CSV 이어쓰기)

위 태그 개수를 확인했다면 실행한다. 진행되는 동안 청크별로 몇 행이 저장됐는지 콘솔에 찍힌다.

In [ ]:
SAVE_DIR = os.path.join(os.getcwd(), "extracted_csv")

csv_path = extractor.extract_day_to_csv(
    TARGET_DATE,
    save_dir=SAVE_DIR,
    target_tags=discovered_tags,
)
csv_path

## 3. 저장된 CSV 갯수 · 용량 확인

`extract_day_to_csv` 가 끝나면서 총 행 수와 파일 크기를 이미 출력하지만, 여기서 한 번 더 파일을 직접 열어 확인한다 (전체를 메모리에 올리지 않고 행 수만 센다).

In [ ]:
if csv_path:
    size_bytes = os.path.getsize(csv_path)
    with open(csv_path, encoding="utf-8-sig") as f:
        n_lines = sum(1 for _ in f)
    n_rows = n_lines - 1  # 헤더 제외

    print(f"파일: {csv_path}")
    print(f"파일 크기: {size_bytes / 1024**2:,.1f} MB")
    print(f"행(row) 수: {n_rows:,}")
    print(f"열(column, 태그) 수: {len(discovered_tags):,}")
else:
    print("저장된 데이터가 없습니다.")